# Benchmark insertion of new predictions
1. Insert 1B rows monotonic ids: ~0.49 hrs
2. Insert 1B rows non-monotonic ids
3. Insert 1000 random rows w/ fk: ~41ms, ~20k r/s
4. Insert 1000 sequential rows w/ fk: 
5. Insert 1000 rows w/ ij index: 
6. Insert 1000 rows w/ z index
7. Insert 1000 rows into 1B-row annotation table w/ B-tree on z-order cell

## Test run: 1 million rows in the primary table, 100,000 rows in the foreign key table.

In [ ]:
import psycopg2
import time

DATABASE_URL = "dbname=testdb user=testuser password=mypassword host=prototyping-pg-1"
TABLE_1_NAME = "proto_2_1_uuid_primary"
TABLE_2_NAME = "proto_2_1_uuid_foreign"
TOTAL_ROWS = 1_000_000
TOTAL_FK_ROWS = 100_000
BATCH_SIZE = 100_000
COMMIT_FREQUENCY = 100

conn = psycopg2.connect(DATABASE_URL)
cur = conn.cursor()

print("Creating tables...")
cur.execute(f"DROP TABLE IF EXISTS {TABLE_2_NAME} CASCADE;")
cur.execute(f"DROP TABLE IF EXISTS {TABLE_1_NAME} CASCADE;")

# Create table with BIGSERIAL primary key
cur.execute(f"""
CREATE TABLE {TABLE_1_NAME} (
    id BIGSERIAL PRIMARY KEY,
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);

CREATE TABLE {TABLE_2_NAME} (
    id BIGSERIAL PRIMARY KEY,
    id_ref BIGINT NOT NULL REFERENCES {TABLE_1_NAME}(id),
    data INT NOT NULL,
    value FLOAT NOT NULL
);
""")
conn.commit()

# Tune PostgreSQL for bulk inserts
cur.execute("SET maintenance_work_mem = '2GB';")
cur.execute("SET work_mem = '512MB';")
cur.execute("SET synchronous_commit = OFF;")
conn.commit()

print("\n=== Phase 1: Insert Rows ===\n")

rows_inserted = 0
start_time = time.time()
batch_count = 0

try:
    cur.execute("BEGIN;")
    
    while rows_inserted < TOTAL_ROWS:
        # Insert simple rows
        cur.execute(f"""
            INSERT INTO {TABLE_1_NAME} (created_at)
            SELECT CURRENT_TIMESTAMP
            FROM generate_series(1, {BATCH_SIZE});
        """)
        
        rows_inserted += BATCH_SIZE
        batch_count += 1
        
        if batch_count % COMMIT_FREQUENCY == 0:
            conn.commit()
            elapsed = time.time() - start_time
            rate = rows_inserted / elapsed
            remaining = TOTAL_ROWS - rows_inserted
            eta_sec = remaining / rate if rate > 0 else 0
            print(f"Inserted {rows_inserted:,} rows in {elapsed:.2f}s ({rate:.0f} rows/sec) - ETA: {eta_sec/3600:.2f}h")
            cur.execute("BEGIN;")
    
    conn.commit()
    
except Exception as e:
    conn.rollback()
    print(f"Error Phase 1: {e}")
    raise

elapsed_phase1 = time.time() - start_time
rate_phase1 = rows_inserted / elapsed_phase1

print(f"\nPhase 1 Complete:")
print(f"  Total rows: {rows_inserted:,}")
print(f"  Elapsed time: {elapsed_phase1:.2f}s")
print(f"  Throughput: {rate_phase1:.0f} rows/sec")
print(f"  Est. time for 1B: {1_000_000_000 / rate_phase1 / 3600:.2f} hours")

print("\n=== Phase 2: Insert FK Rows (Sequential) ===\n")

rows_inserted_fk = 0
start_time_fk = time.time()
batch_count_fk = 0

try:
    cur.execute("BEGIN;")
    
    while rows_inserted_fk < TOTAL_FK_ROWS:
        # Sample from primary table sequentially
        cur.execute(f"""
            INSERT INTO {TABLE_2_NAME} (id_ref, data, value)
            SELECT 
                id,
                row_number() OVER() % 1000,
                (row_number() OVER() - 1) * 0.5
            FROM (
                SELECT id FROM {TABLE_1_NAME} 
                ORDER BY id
                LIMIT {BATCH_SIZE}
                OFFSET {rows_inserted_fk}
            ) t;
        """)
        
        rows_inserted_fk += BATCH_SIZE
        batch_count_fk += 1
        
        if batch_count_fk % COMMIT_FREQUENCY == 0:
            conn.commit()
            elapsed = time.time() - start_time_fk
            rate = rows_inserted_fk / elapsed
            print(f"Inserted {rows_inserted_fk:,} rows in {elapsed:.2f}s ({rate:.0f} rows/sec)")
            cur.execute("BEGIN;")
    
    conn.commit()
        
    print("Creating index...")
    start_index_time = time.time()
    cur.execute(f"CREATE INDEX idx_id_ref ON {TABLE_2_NAME}(id_ref);")
    conn.commit()
    elapsed_index = time.time() - start_index_time
    print(f"Index created in {elapsed_index:.2f}s")
    conn.commit()
    
except Exception as e:
    conn.rollback()
    print(f"Error Phase 2: {e}")
    raise

elapsed_phase2 = time.time() - start_time_fk
rate_phase2 = rows_inserted_fk / elapsed_phase2 if elapsed_phase2 > 0 else 0

print(f"\nPhase 2 Complete:")
print(f"  Total rows: {rows_inserted_fk:,}")
print(f"  Elapsed time: {elapsed_phase2:.2f}s")
print(f"  Throughput: {rate_phase2:.0f} rows/sec")

total_elapsed = elapsed_phase1 + elapsed_phase2
total_rows = rows_inserted + rows_inserted_fk
overall_rate = total_rows / total_elapsed

print("\n" + "="*60)
print("BENCHMARK SUMMARY")
print("="*60)
print(f"Phase 1 (Insert): {rows_inserted:,} rows in {elapsed_phase1:.2f}s ({rate_phase1:.0f} rows/sec)")
print(f"Phase 2 (Insert): {rows_inserted_fk:,} rows in {elapsed_phase2:.2f}s ({rate_phase2:.0f} rows/sec)")
print(f"\nTotal: {total_rows:,} rows in {total_elapsed:.2f}s ({overall_rate:.0f} rows/sec)")
print(f"Est. time for 1B rows (Phase 1): {1_000_000_000 / rate_phase1 / 3600:.2f} hours")
print("="*60)

cur.close()
conn.close()


## Insert 1B rows monotonic ids

In [1]:
import psycopg2
import time
import random

DATABASE_URL = "dbname=testdb user=testuser password=mypassword host=prototyping-pg-1"
TABLE_1_NAME = "b_rows_monotonic_ids"
TABLE_2_NAME = "onek_rows_with_fk"
TOTAL_ROWS = 1_000_000_000       # 1 billion primary rows
BATCH_SIZE = 100_000
COMMIT_FREQUENCY = 100
PREDICTION_BATCH_SIZE = 1_000    # batch size for the FK insert benchmark
PREDICTION_RUNS = 10             # repeat N times to get a stable average

conn = psycopg2.connect(DATABASE_URL)
cur = conn.cursor()

# Create tables only if they don't already exist
cur.execute(f"""
CREATE TABLE IF NOT EXISTS {TABLE_1_NAME} (
    id BIGSERIAL PRIMARY KEY,
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);

CREATE TABLE IF NOT EXISTS {TABLE_2_NAME} (
    id BIGSERIAL PRIMARY KEY,
    id_ref BIGINT NOT NULL REFERENCES {TABLE_1_NAME}(id),
    data INT NOT NULL,
    value FLOAT NOT NULL
);
""")
conn.commit()

# Tune PostgreSQL for bulk inserts
cur.execute("SET maintenance_work_mem = '4GB';")
cur.execute("SET work_mem = '1GB';")
cur.execute("SET synchronous_commit = OFF;")
conn.commit()

# Check whether the primary table already has data
cur.execute(f"SELECT COUNT(*) FROM {TABLE_1_NAME};")
table_1_row_count = cur.fetchone()[0]

# ─────────────────────────────────────────────
# Phase 1: Insert 1 billion rows (primary table)
# ─────────────────────────────────────────────
if table_1_row_count > 0:
    print(f"Skipping Phase 1 — {TABLE_1_NAME} already exists with {table_1_row_count:,} rows.")
    rows_inserted = table_1_row_count
    elapsed_phase1 = 0
    rate_phase1 = float('nan')
else:
    print("\n=== Phase 1: Insert 1 Billion Rows (Primary Table) ===\n")

    rows_inserted = 0
    start_time = time.time()
    batch_count = 0

    try:
        cur.execute("BEGIN;")

        while rows_inserted < TOTAL_ROWS:
            cur.execute(f"""
                INSERT INTO {TABLE_1_NAME} (created_at)
                SELECT CURRENT_TIMESTAMP
                FROM generate_series(1, {BATCH_SIZE});
            """)

            rows_inserted += BATCH_SIZE
            batch_count += 1

            if batch_count % COMMIT_FREQUENCY == 0:
                conn.commit()
                elapsed = time.time() - start_time
                rate = rows_inserted / elapsed
                remaining = TOTAL_ROWS - rows_inserted
                eta_sec = remaining / rate if rate > 0 else 0
                print(
                    f"  {rows_inserted:>15,} rows | {elapsed:>8.2f}s | "
                    f"{rate:>10,.0f} rows/s | ETA: {eta_sec / 3600:.2f}h"
                )
                cur.execute("BEGIN;")

        conn.commit()

    except Exception as e:
        conn.rollback()
        print(f"Error Phase 1: {e}")
        raise

    elapsed_phase1 = time.time() - start_time
    rate_phase1 = rows_inserted / elapsed_phase1

    print(f"\nPhase 1 Complete:")
    print(f"  Total rows : {rows_inserted:,}")
    print(f"  Elapsed    : {elapsed_phase1:.2f}s")
    print(f"  Throughput : {rate_phase1:,.0f} rows/sec")



=== Phase 1: Insert 1 Billion Rows (Primary Table) ===

       10,000,000 rows |    15.87s |    630,082 rows/s | ETA: 0.44h
       20,000,000 rows |    30.72s |    650,989 rows/s | ETA: 0.42h
       30,000,000 rows |    48.29s |    621,204 rows/s | ETA: 0.43h
       40,000,000 rows |    65.46s |    611,019 rows/s | ETA: 0.44h
       50,000,000 rows |    79.14s |    631,762 rows/s | ETA: 0.42h
       60,000,000 rows |    93.41s |    642,318 rows/s | ETA: 0.41h
       70,000,000 rows |   110.77s |    631,934 rows/s | ETA: 0.41h
       80,000,000 rows |   128.62s |    621,979 rows/s | ETA: 0.41h
       90,000,000 rows |   144.72s |    621,873 rows/s | ETA: 0.41h
      100,000,000 rows |   162.30s |    616,129 rows/s | ETA: 0.41h
      110,000,000 rows |   179.72s |    612,057 rows/s | ETA: 0.40h
      120,000,000 rows |   196.06s |    612,067 rows/s | ETA: 0.40h
      130,000,000 rows |   210.66s |    617,112 rows/s | ETA: 0.39h
      140,000,000 rows |   225.78s |    620,081 rows/s | ET

## Insert 1B rows non-monotonic ids

In [ ]:
import psycopg2
import time
import random

DATABASE_URL = "dbname=testdb user=testuser password=mypassword host=prototyping-pg-1"
TABLE_1_NAME = "b_rows_non_monotonic_ids"
TOTAL_ROWS = 1_000_000_000       # 1 billion primary rows
BATCH_SIZE = 100_000
COMMIT_FREQUENCY = 100
PREDICTION_BATCH_SIZE = 1_000    # batch size for the FK insert benchmark
PREDICTION_RUNS = 10             # repeat N times to get a stable average

conn = psycopg2.connect(DATABASE_URL)
cur = conn.cursor()

# Create tables only if they don't already exist
cur.execute(f"""
CREATE TABLE IF NOT EXISTS {TABLE_1_NAME} (
    id BIGINT PRIMARY KEY,
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);

""")
conn.commit()

# Tune PostgreSQL for bulk inserts
cur.execute("SET maintenance_work_mem = '4GB';")
cur.execute("SET work_mem = '1GB';")
cur.execute("SET synchronous_commit = OFF;")
conn.commit()

# Check whether the primary table already has data
cur.execute(f"SELECT COUNT(*) FROM {TABLE_1_NAME};")
table_1_row_count = cur.fetchone()[0]

# ─────────────────────────────────────────────
# Phase 1: Insert 1 billion rows (primary table)
# ─────────────────────────────────────────────
if table_1_row_count > 0:
    print(f"Skipping Phase 1 — {TABLE_1_NAME} already exists with {table_1_row_count:,} rows.")
    rows_inserted = table_1_row_count
    elapsed_phase1 = 0
    rate_phase1 = float('nan')
else:
    print("\n=== Phase 1: Insert 1 Billion Rows (Primary Table) ===\n")

    rows_inserted = 0
    start_time = time.time()
    batch_count = 0

    try:
        cur.execute("BEGIN;")

        while rows_inserted < TOTAL_ROWS:
            cur.execute(f"""
                INSERT INTO {TABLE_1_NAME} (created_at)
                SELECT CURRENT_TIMESTAMP
                FROM generate_series(1, {BATCH_SIZE});
            """)

            rows_inserted += BATCH_SIZE
            batch_count += 1

            if batch_count % COMMIT_FREQUENCY == 0:
                conn.commit()
                elapsed = time.time() - start_time
                rate = rows_inserted / elapsed
                remaining = TOTAL_ROWS - rows_inserted
                eta_sec = remaining / rate if rate > 0 else 0
                print(
                    f"  {rows_inserted:>15,} rows | {elapsed:>8.2f}s | "
                    f"{rate:>10,.0f} rows/s | ETA: {eta_sec / 3600:.2f}h"
                )
                cur.execute("BEGIN;")

        conn.commit()

    except Exception as e:
        conn.rollback()
        print(f"Error Phase 1: {e}")
        raise

    elapsed_phase1 = time.time() - start_time
    rate_phase1 = rows_inserted / elapsed_phase1

    print(f"\nPhase 1 Complete:")
    print(f"  Total rows : {rows_inserted:,}")
    print(f"  Elapsed    : {elapsed_phase1:.2f}s")
    print(f"  Throughput : {rate_phase1:,.0f} rows/sec")


## Insert 1000 random rows w/ fk

In [4]:

# ──────────────────────────────────────────────────────────────────────────────
# Phase 2: Benchmark — insert 1,000 FK rows into empty TABLE_2 (repeated N times)
# ──────────────────────────────────────────────────────────────────────────────
print(f"\n=== Phase 2: Insert {PREDICTION_BATCH_SIZE:,} FK Rows into Empty Table (×{PREDICTION_RUNS} runs) ===\n")

latencies = []

for run in range(1, PREDICTION_RUNS + 1):
    # Truncate the FK table to ensure each run starts from empty
    cur.execute(f"TRUNCATE TABLE {TABLE_2_NAME} RESTART IDENTITY;")
    conn.commit()

    t0 = time.time()
    cur.execute(f"""
        INSERT INTO {TABLE_2_NAME} (id_ref, data, value)
        SELECT
            floor(random() * {rows_inserted})::BIGINT + 1,
            (random() * 1000)::INT,
            random() * 100.0
        FROM generate_series(1, {PREDICTION_BATCH_SIZE});
    """)
    conn.commit()
    latency_ms = (time.time() - t0) * 1000
    latencies.append(latency_ms)
    print(f"  Run {run:>2}/{PREDICTION_RUNS}: {latency_ms:.2f} ms  ({PREDICTION_BATCH_SIZE / (latency_ms / 1000):,.0f} rows/sec)")

avg_ms   = sum(latencies) / len(latencies)
min_ms   = min(latencies)
max_ms   = max(latencies)
avg_rate = PREDICTION_BATCH_SIZE / (avg_ms / 1000)

print(f"\n  Avg latency : {avg_ms:.2f} ms")
print(f"  Min latency : {min_ms:.2f} ms")
print(f"  Max latency : {max_ms:.2f} ms")
print(f"  Avg rate    : {avg_rate:,.0f} rows/sec")



=== Phase 2: Insert 1,000 FK Rows into Empty Table (×10 runs) ===

  Run  1/10: 207.09 ms  (4,829 rows/sec)
  Run  2/10: 197.30 ms  (5,068 rows/sec)
  Run  3/10: 210.93 ms  (4,741 rows/sec)
  Run  4/10: 210.93 ms  (4,741 rows/sec)
  Run  5/10: 209.37 ms  (4,776 rows/sec)
  Run  6/10: 213.02 ms  (4,694 rows/sec)
  Run  7/10: 221.39 ms  (4,517 rows/sec)
  Run  8/10: 195.01 ms  (5,128 rows/sec)
  Run  9/10: 180.46 ms  (5,541 rows/sec)
  Run 10/10: 177.04 ms  (5,648 rows/sec)

  Avg latency : 202.25 ms
  Min latency : 177.04 ms
  Max latency : 221.39 ms
  Avg rate    : 4,944 rows/sec


## Insert 1000 sequential rows w/ fk

In [5]:
import psycopg2
import time
import random

DATABASE_URL = "dbname=testdb user=testuser password=mypassword host=prototyping-pg-1"
TABLE_1_NAME = "b_rows_monotonic_ids"
TABLE_2_NAME_SEQ = "onek_rows_with_fk_sequential"
PREDICTION_BATCH_SIZE = 1_000
PREDICTION_RUNS = 10

conn_seq = psycopg2.connect(DATABASE_URL)
cur_seq = conn_seq.cursor()

# Get the current row count of the primary table
cur_seq.execute(f"SELECT COUNT(*) FROM {TABLE_1_NAME};")
rows_in_primary = cur_seq.fetchone()[0]

# Create FK table if it doesn't exist
cur_seq.execute(f"""
CREATE TABLE IF NOT EXISTS {TABLE_2_NAME_SEQ} (
    id BIGSERIAL PRIMARY KEY,
    id_ref BIGINT NOT NULL REFERENCES {TABLE_1_NAME}(id),
    data INT NOT NULL,
    value FLOAT NOT NULL
);
""")
conn_seq.commit()

print(f"\n=== Benchmark: Insert {PREDICTION_BATCH_SIZE:,} Sequential FK Rows (×{PREDICTION_RUNS} runs) ===\n")

seq_latencies = []

for run in range(1, PREDICTION_RUNS + 1):
    cur_seq.execute(f"TRUNCATE TABLE {TABLE_2_NAME_SEQ} RESTART IDENTITY;")
    conn_seq.commit()

    # Pick a random start offset for variety across runs
    start_id = random.randint(1, max(1, rows_in_primary - PREDICTION_BATCH_SIZE))

    t0 = time.time()
    cur_seq.execute(f"""
        INSERT INTO {TABLE_2_NAME_SEQ} (id_ref, data, value)
        SELECT
            {start_id}::BIGINT + gs - 1,
            (random() * 1000)::INT,
            random() * 100.0
        FROM generate_series(1, {PREDICTION_BATCH_SIZE}) gs;
    """)
    conn_seq.commit()
    latency_ms = (time.time() - t0) * 1000
    seq_latencies.append(latency_ms)
    print(f"  Run {run:>2}/{PREDICTION_RUNS}: {latency_ms:.2f} ms  ({PREDICTION_BATCH_SIZE / (latency_ms / 1000):,.0f} rows/sec)  [start_id={start_id:,}]")

seq_avg_ms   = sum(seq_latencies) / len(seq_latencies)
seq_min_ms   = min(seq_latencies)
seq_max_ms   = max(seq_latencies)
seq_avg_rate = PREDICTION_BATCH_SIZE / (seq_avg_ms / 1000)

print(f"\n  Avg latency : {seq_avg_ms:.2f} ms")
print(f"  Min latency : {seq_min_ms:.2f} ms")
print(f"  Max latency : {seq_max_ms:.2f} ms")
print(f"  Avg rate    : {seq_avg_rate:,.0f} rows/sec")

cur_seq.close()
conn_seq.close()



=== Benchmark: Insert 1,000 Sequential FK Rows (×10 runs) ===

  Run  1/10: 11.26 ms  (88,840 rows/sec)  [start_id=891,404,002]
  Run  2/10: 7.91 ms  (126,472 rows/sec)  [start_id=568,859,566]
  Run  3/10: 7.82 ms  (127,832 rows/sec)  [start_id=809,142,012]
  Run  4/10: 7.73 ms  (129,298 rows/sec)  [start_id=408,488,319]
  Run  5/10: 7.51 ms  (133,186 rows/sec)  [start_id=935,182,423]
  Run  6/10: 7.84 ms  (127,572 rows/sec)  [start_id=553,202,984]
  Run  7/10: 7.89 ms  (126,808 rows/sec)  [start_id=353,364,536]
  Run  8/10: 7.65 ms  (130,761 rows/sec)  [start_id=144,407,250]
  Run  9/10: 8.08 ms  (123,806 rows/sec)  [start_id=791,758,693]
  Run 10/10: 10.18 ms  (98,230 rows/sec)  [start_id=790,271,103]

  Avg latency : 8.39 ms
  Min latency : 7.51 ms
  Max latency : 11.26 ms
  Avg rate    : 119,250 rows/sec


In [6]:
# ─────────────────
# Summary
# ─────────────────
print("\n" + "=" * 70)
print("BENCHMARK SUMMARY")
print("=" * 70)
phase1_summary = (
    f"skipped ({rows_inserted:,} rows pre-existing)"
    if elapsed_phase1 == 0
    else f"{elapsed_phase1:.2f}s  |  {rate_phase1:,.0f} rows/sec"
)
print(f"Phase 1 (Primary, 1B rows)                : {phase1_summary}")
print(f"Phase 2 (1k random FK rows, empty table)  : avg {avg_ms:.2f} ms  |  {avg_rate:,.0f} rows/sec")
print(f"  Min: {min_ms:.2f} ms  |  Max: {max_ms:.2f} ms")
print(f"Phase 3 (1k sequential FK rows, empty tbl): avg {seq_avg_ms:.2f} ms  |  {seq_avg_rate:,.0f} rows/sec")
print(f"  Min: {seq_min_ms:.2f} ms  |  Max: {seq_max_ms:.2f} ms")
print("=" * 70)

cur.close()
conn.close()



BENCHMARK SUMMARY
Phase 1 (Primary, 1B rows)                : 1697.26s  |  589,186 rows/sec
Phase 2 (1k random FK rows, empty table)  : avg 202.25 ms  |  4,944 rows/sec
  Min: 177.04 ms  |  Max: 221.39 ms
Phase 3 (1k sequential FK rows, empty tbl): avg 8.39 ms  |  119,250 rows/sec
  Min: 7.51 ms  |  Max: 11.26 ms


## Insert 1000 rows w/ random ij index

In [1]:

import psycopg2
import psycopg2.extras
import time
import random
import sys

sys.path.insert(0, '/opt/PatchSorter/prototyping')
from utils import HierarchicalGridIndexIJ

DATABASE_URL = "dbname=testdb user=testuser password=mypassword host=prototyping-pg-1"
TABLE_IJ_NAME = "onek_rows_ij_index"
PREDICTION_BATCH_SIZE = 1_000
PREDICTION_RUNS = 10
IJ_LEVEL = 10
SPATIAL_EXTENT = 1_000_000.0

conn_ij = psycopg2.connect(DATABASE_URL)
cur_ij = conn_ij.cursor()

cur_ij.execute(f"""
CREATE TABLE IF NOT EXISTS {TABLE_IJ_NAME} (
    id      BIGSERIAL PRIMARY KEY,
    id_ref  BIGINT NOT NULL,
    data    INT NOT NULL,
    value   FLOAT NOT NULL,
    ij_cell BIGINT NOT NULL
);
""")
cur_ij.execute(f"CREATE INDEX IF NOT EXISTS idx_ij_cell ON {TABLE_IJ_NAME}(ij_cell);")
conn_ij.commit()

grid_ij = HierarchicalGridIndexIJ(cell_size=1.0)

print(f"\n=== Benchmark: Insert {PREDICTION_BATCH_SIZE:,} IJ-indexed Rows (×{PREDICTION_RUNS} runs) ===")
print(f"  IJ level : {IJ_LEVEL}  (cell size = 1/{2**IJ_LEVEL})\n")

ij_latencies = []

for run in range(1, PREDICTION_RUNS + 1):
    cur_ij.execute(f"TRUNCATE TABLE {TABLE_IJ_NAME} RESTART IDENTITY;")
    conn_ij.commit()

    # Pre-generate rows — cell computation excluded from INSERT timing
    rows = [
        (
            random.randint(1, 1_000_000_000),
            random.randint(0, 1000),
            random.uniform(0, 100.0),
            grid_ij.point_to_cell(
                random.uniform(0, SPATIAL_EXTENT),
                random.uniform(0, SPATIAL_EXTENT),
                IJ_LEVEL,
            ),
        )
        for _ in range(PREDICTION_BATCH_SIZE)
    ]

    t0 = time.time()
    psycopg2.extras.execute_values(
        cur_ij,
        f"INSERT INTO {TABLE_IJ_NAME} (id_ref, data, value, ij_cell) VALUES %s",
        rows,
    )
    conn_ij.commit()
    latency_ms = (time.time() - t0) * 1000
    ij_latencies.append(latency_ms)
    print(f"  Run {run:>2}/{PREDICTION_RUNS}: {latency_ms:.2f} ms  ({PREDICTION_BATCH_SIZE / (latency_ms / 1000):,.0f} rows/sec)")

ij_avg_ms   = sum(ij_latencies) / len(ij_latencies)
ij_min_ms   = min(ij_latencies)
ij_max_ms   = max(ij_latencies)
ij_avg_rate = PREDICTION_BATCH_SIZE / (ij_avg_ms / 1000)

print(f"\n  Avg latency : {ij_avg_ms:.2f} ms")
print(f"  Min latency : {ij_min_ms:.2f} ms")
print(f"  Max latency : {ij_max_ms:.2f} ms")
print(f"  Avg rate    : {ij_avg_rate:,.0f} rows/sec")

cur_ij.close()
conn_ij.close()



=== Benchmark: Insert 1,000 IJ-indexed Rows (×10 runs) ===
  IJ level : 10  (cell size = 1/1024)

  Run  1/10: 21.88 ms  (45,695 rows/sec)
  Run  2/10: 13.78 ms  (72,592 rows/sec)
  Run  3/10: 16.16 ms  (61,883 rows/sec)
  Run  4/10: 12.50 ms  (80,023 rows/sec)
  Run  5/10: 13.30 ms  (75,210 rows/sec)
  Run  6/10: 12.51 ms  (79,920 rows/sec)
  Run  7/10: 11.83 ms  (84,534 rows/sec)
  Run  8/10: 18.55 ms  (53,904 rows/sec)
  Run  9/10: 13.01 ms  (76,888 rows/sec)
  Run 10/10: 12.43 ms  (80,432 rows/sec)

  Avg latency : 14.59 ms
  Min latency : 11.83 ms
  Max latency : 21.88 ms
  Avg rate    : 68,519 rows/sec


## Insert 1000 rows w/ ij index + FK

In [1]:

import psycopg2
import psycopg2.extras
import time
import random
import sys

sys.path.insert(0, '/opt/PatchSorter/prototyping')
from utils import HierarchicalGridIndexIJ

DATABASE_URL = "dbname=testdb user=testuser password=mypassword host=prototyping-pg-1"
TABLE_1_NAME      = "b_rows_monotonic_ids"
TABLE_IJ_FK_NAME  = "onek_rows_ij_index_fk"
PREDICTION_BATCH_SIZE = 1_000
PREDICTION_RUNS = 10
IJ_LEVEL = 10
SPATIAL_EXTENT = 1_000_000.0

conn_ij_fk = psycopg2.connect(DATABASE_URL)
cur_ij_fk = conn_ij_fk.cursor()

cur_ij_fk.execute(f"SELECT COUNT(*) FROM {TABLE_1_NAME};")
rows_in_primary_ij_fk = cur_ij_fk.fetchone()[0]

cur_ij_fk.execute(f"""
CREATE TABLE IF NOT EXISTS {TABLE_IJ_FK_NAME} (
    id      BIGSERIAL PRIMARY KEY,
    id_ref  BIGINT NOT NULL REFERENCES {TABLE_1_NAME}(id),
    data    INT NOT NULL,
    value   FLOAT NOT NULL,
    ij_cell BIGINT NOT NULL
);
""")
cur_ij_fk.execute(f"CREATE INDEX IF NOT EXISTS idx_ij_cell_fk ON {TABLE_IJ_FK_NAME}(ij_cell);")
conn_ij_fk.commit()

grid_ij_fk = HierarchicalGridIndexIJ(cell_size=1.0)

print(f"\n=== Benchmark: Insert {PREDICTION_BATCH_SIZE:,} IJ-indexed Rows w/ FK (×{PREDICTION_RUNS} runs) ===")
print(f"  Primary table : {rows_in_primary_ij_fk:,} rows")
print(f"  IJ level      : {IJ_LEVEL}  (cell size = 1/{2**IJ_LEVEL})\n")

ij_fk_latencies = []

for run in range(1, PREDICTION_RUNS + 1):
    cur_ij_fk.execute(f"TRUNCATE TABLE {TABLE_IJ_FK_NAME} RESTART IDENTITY;")
    conn_ij_fk.commit()

    # Pre-generate rows — cell computation excluded from INSERT timing
    rows = [
        (
            random.randint(1, rows_in_primary_ij_fk),
            random.randint(0, 1000),
            random.uniform(0, 100.0),
            grid_ij_fk.point_to_cell(
                random.uniform(0, SPATIAL_EXTENT),
                random.uniform(0, SPATIAL_EXTENT),
                IJ_LEVEL,
            ),
        )
        for _ in range(PREDICTION_BATCH_SIZE)
    ]

    t0 = time.time()
    psycopg2.extras.execute_values(
        cur_ij_fk,
        f"INSERT INTO {TABLE_IJ_FK_NAME} (id_ref, data, value, ij_cell) VALUES %s",
        rows,
    )
    conn_ij_fk.commit()
    latency_ms = (time.time() - t0) * 1000
    ij_fk_latencies.append(latency_ms)
    print(f"  Run {run:>2}/{PREDICTION_RUNS}: {latency_ms:.2f} ms  ({PREDICTION_BATCH_SIZE / (latency_ms / 1000):,.0f} rows/sec)")

ij_fk_avg_ms   = sum(ij_fk_latencies) / len(ij_fk_latencies)
ij_fk_min_ms   = min(ij_fk_latencies)
ij_fk_max_ms   = max(ij_fk_latencies)
ij_fk_avg_rate = PREDICTION_BATCH_SIZE / (ij_fk_avg_ms / 1000)

print(f"\n  Avg latency : {ij_fk_avg_ms:.2f} ms")
print(f"  Min latency : {ij_fk_min_ms:.2f} ms")
print(f"  Max latency : {ij_fk_max_ms:.2f} ms")
print(f"  Avg rate    : {ij_fk_avg_rate:,.0f} rows/sec")

cur_ij_fk.close()
conn_ij_fk.close()



=== Benchmark: Insert 1,000 IJ-indexed Rows w/ FK (×10 runs) ===
  Primary table : 1,000,000,000 rows
  IJ level      : 10  (cell size = 1/1024)

  Run  1/10: 222.87 ms  (4,487 rows/sec)
  Run  2/10: 289.84 ms  (3,450 rows/sec)
  Run  3/10: 229.71 ms  (4,353 rows/sec)
  Run  4/10: 232.70 ms  (4,297 rows/sec)
  Run  5/10: 223.49 ms  (4,474 rows/sec)
  Run  6/10: 236.40 ms  (4,230 rows/sec)
  Run  7/10: 231.71 ms  (4,316 rows/sec)
  Run  8/10: 211.10 ms  (4,737 rows/sec)
  Run  9/10: 201.50 ms  (4,963 rows/sec)
  Run 10/10: 194.92 ms  (5,130 rows/sec)

  Avg latency : 227.42 ms
  Min latency : 194.92 ms
  Max latency : 289.84 ms
  Avg rate    : 4,397 rows/sec


## Insert 1000 rows w/ random z index

In [2]:

import psycopg2
import psycopg2.extras
import time
import random
import sys

sys.path.insert(0, '/opt/PatchSorter/prototyping')
from utils import HierarchicalGridIndexZOrder

DATABASE_URL = "dbname=testdb user=testuser password=mypassword host=prototyping-pg-1"
TABLE_Z_NAME = "onek_rows_z_index"
PREDICTION_BATCH_SIZE = 1_000
PREDICTION_RUNS = 10
Z_LEVEL = 10
SPATIAL_EXTENT = 1_000_000.0

conn_z = psycopg2.connect(DATABASE_URL)
cur_z = conn_z.cursor()

cur_z.execute(f"""
CREATE TABLE IF NOT EXISTS {TABLE_Z_NAME} (
    id     BIGSERIAL PRIMARY KEY,
    id_ref BIGINT NOT NULL,
    data   INT NOT NULL,
    value  FLOAT NOT NULL,
    z_cell BIGINT NOT NULL
);
""")
cur_z.execute(f"CREATE INDEX IF NOT EXISTS idx_z_cell ON {TABLE_Z_NAME}(z_cell);")
conn_z.commit()

grid_z = HierarchicalGridIndexZOrder(cell_size=1.0)

print(f"\n=== Benchmark: Insert {PREDICTION_BATCH_SIZE:,} Z-Order-indexed Rows (×{PREDICTION_RUNS} runs) ===")
print(f"  Z-order level : {Z_LEVEL}  (cell size = 1/{2**Z_LEVEL})\n")

z_latencies = []

for run in range(1, PREDICTION_RUNS + 1):
    cur_z.execute(f"TRUNCATE TABLE {TABLE_Z_NAME} RESTART IDENTITY;")
    conn_z.commit()

    # Pre-generate rows — cell computation excluded from INSERT timing
    rows = [
        (
            random.randint(1, 1_000_000_000),
            random.randint(0, 1000),
            random.uniform(0, 100.0),
            grid_z.point_to_cell(
                random.uniform(0, SPATIAL_EXTENT),
                random.uniform(0, SPATIAL_EXTENT),
                Z_LEVEL,
            ),
        )
        for _ in range(PREDICTION_BATCH_SIZE)
    ]

    t0 = time.time()
    psycopg2.extras.execute_values(
        cur_z,
        f"INSERT INTO {TABLE_Z_NAME} (id_ref, data, value, z_cell) VALUES %s",
        rows,
    )
    conn_z.commit()
    latency_ms = (time.time() - t0) * 1000
    z_latencies.append(latency_ms)
    print(f"  Run {run:>2}/{PREDICTION_RUNS}: {latency_ms:.2f} ms  ({PREDICTION_BATCH_SIZE / (latency_ms / 1000):,.0f} rows/sec)")

z_avg_ms   = sum(z_latencies) / len(z_latencies)
z_min_ms   = min(z_latencies)
z_max_ms   = max(z_latencies)
z_avg_rate = PREDICTION_BATCH_SIZE / (z_avg_ms / 1000)

print(f"\n  Avg latency : {z_avg_ms:.2f} ms")
print(f"  Min latency : {z_min_ms:.2f} ms")
print(f"  Max latency : {z_max_ms:.2f} ms")
print(f"  Avg rate    : {z_avg_rate:,.0f} rows/sec")

cur_z.close()
conn_z.close()



=== Benchmark: Insert 1,000 Z-Order-indexed Rows (×10 runs) ===
  Z-order level : 10  (cell size = 1/1024)

  Run  1/10: 15.22 ms  (65,712 rows/sec)
  Run  2/10: 16.13 ms  (62,010 rows/sec)
  Run  3/10: 18.42 ms  (54,287 rows/sec)
  Run  4/10: 17.45 ms  (57,295 rows/sec)
  Run  5/10: 17.25 ms  (57,988 rows/sec)
  Run  6/10: 15.01 ms  (66,637 rows/sec)
  Run  7/10: 13.59 ms  (73,579 rows/sec)
  Run  8/10: 12.57 ms  (79,532 rows/sec)
  Run  9/10: 21.16 ms  (47,259 rows/sec)
  Run 10/10: 16.30 ms  (61,354 rows/sec)

  Avg latency : 16.31 ms
  Min latency : 12.57 ms
  Max latency : 21.16 ms
  Avg rate    : 61,315 rows/sec


## Insert 1000 rows w/ z index + FK

In [2]:

import psycopg2
import psycopg2.extras
import time
import random
import sys

sys.path.insert(0, '/opt/PatchSorter/prototyping')
from utils import HierarchicalGridIndexZOrder

DATABASE_URL = "dbname=testdb user=testuser password=mypassword host=prototyping-pg-1"
TABLE_1_NAME     = "b_rows_monotonic_ids"
TABLE_Z_FK_NAME  = "onek_rows_z_index_fk"
PREDICTION_BATCH_SIZE = 1_000
PREDICTION_RUNS = 10
Z_LEVEL = 10
SPATIAL_EXTENT = 1_000_000.0

conn_z_fk = psycopg2.connect(DATABASE_URL)
cur_z_fk = conn_z_fk.cursor()

cur_z_fk.execute(f"SELECT COUNT(*) FROM {TABLE_1_NAME};")
rows_in_primary_z_fk = cur_z_fk.fetchone()[0]

cur_z_fk.execute(f"""
CREATE TABLE IF NOT EXISTS {TABLE_Z_FK_NAME} (
    id     BIGSERIAL PRIMARY KEY,
    id_ref BIGINT NOT NULL REFERENCES {TABLE_1_NAME}(id),
    data   INT NOT NULL,
    value  FLOAT NOT NULL,
    z_cell BIGINT NOT NULL
);
""")
cur_z_fk.execute(f"CREATE INDEX IF NOT EXISTS idx_z_cell_fk ON {TABLE_Z_FK_NAME}(z_cell);")
conn_z_fk.commit()

grid_z_fk = HierarchicalGridIndexZOrder(cell_size=1.0)

print(f"\n=== Benchmark: Insert {PREDICTION_BATCH_SIZE:,} Z-Order-indexed Rows w/ FK (×{PREDICTION_RUNS} runs) ===")
print(f"  Primary table : {rows_in_primary_z_fk:,} rows")
print(f"  Z-order level : {Z_LEVEL}  (cell size = 1/{2**Z_LEVEL})\n")

z_fk_latencies = []

for run in range(1, PREDICTION_RUNS + 1):
    cur_z_fk.execute(f"TRUNCATE TABLE {TABLE_Z_FK_NAME} RESTART IDENTITY;")
    conn_z_fk.commit()

    # Pre-generate rows — cell computation excluded from INSERT timing
    rows = [
        (
            random.randint(1, rows_in_primary_z_fk),
            random.randint(0, 1000),
            random.uniform(0, 100.0),
            grid_z_fk.point_to_cell(
                random.uniform(0, SPATIAL_EXTENT),
                random.uniform(0, SPATIAL_EXTENT),
                Z_LEVEL,
            ),
        )
        for _ in range(PREDICTION_BATCH_SIZE)
    ]

    t0 = time.time()
    psycopg2.extras.execute_values(
        cur_z_fk,
        f"INSERT INTO {TABLE_Z_FK_NAME} (id_ref, data, value, z_cell) VALUES %s",
        rows,
    )
    conn_z_fk.commit()
    latency_ms = (time.time() - t0) * 1000
    z_fk_latencies.append(latency_ms)
    print(f"  Run {run:>2}/{PREDICTION_RUNS}: {latency_ms:.2f} ms  ({PREDICTION_BATCH_SIZE / (latency_ms / 1000):,.0f} rows/sec)")

z_fk_avg_ms   = sum(z_fk_latencies) / len(z_fk_latencies)
z_fk_min_ms   = min(z_fk_latencies)
z_fk_max_ms   = max(z_fk_latencies)
z_fk_avg_rate = PREDICTION_BATCH_SIZE / (z_fk_avg_ms / 1000)

print(f"\n  Avg latency : {z_fk_avg_ms:.2f} ms")
print(f"  Min latency : {z_fk_min_ms:.2f} ms")
print(f"  Max latency : {z_fk_max_ms:.2f} ms")
print(f"  Avg rate    : {z_fk_avg_rate:,.0f} rows/sec")

cur_z_fk.close()
conn_z_fk.close()



=== Benchmark: Insert 1,000 Z-Order-indexed Rows w/ FK (×10 runs) ===
  Primary table : 1,000,000,000 rows
  Z-order level : 10  (cell size = 1/1024)

  Run  1/10: 197.03 ms  (5,075 rows/sec)
  Run  2/10: 195.80 ms  (5,107 rows/sec)
  Run  3/10: 185.94 ms  (5,378 rows/sec)
  Run  4/10: 181.43 ms  (5,512 rows/sec)
  Run  5/10: 177.34 ms  (5,639 rows/sec)
  Run  6/10: 191.87 ms  (5,212 rows/sec)
  Run  7/10: 204.25 ms  (4,896 rows/sec)
  Run  8/10: 198.74 ms  (5,032 rows/sec)
  Run  9/10: 182.26 ms  (5,487 rows/sec)
  Run 10/10: 190.97 ms  (5,236 rows/sec)

  Avg latency : 190.56 ms
  Min latency : 177.34 ms
  Max latency : 204.25 ms
  Avg rate    : 5,248 rows/sec


## Insert 1000 pred_patches into 1B-row table w/ B-tree on grid_cell_id

In [ ]:
import psycopg2
import psycopg2.extras
import time
import random
import sys

sys.path.insert(0, '/opt/PatchSorter/prototyping')
from utils import HierarchicalGridIndexZOrder

DATABASE_URL          = "dbname=testdb user=testuser password=mypassword host=prototyping-pg-1"
TABLE_PRED_PATCHES    = "b_pred_patches"
TABLE_GRID_CELLS      = "b_grid_cells"
TABLE_LABEL_CLASSES   = "b_label_classes"
TOTAL_PRED_ROWS       = 1_000_000_000
ANNOT_BATCH_SIZE      = 100_000
ANNOT_COMMIT_FREQ     = 100
PREDICTION_BATCH_SIZE = 1_000
PREDICTION_RUNS       = 10
Z_LEVEL               = 10
SPATIAL_EXTENT        = 1_000_000.0
NUM_LABEL_CLASSES     = 100
NUM_GRID_CELLS        = 4 ** Z_LEVEL   # 2^20 = 1,048,576 cells at level 10

conn_pp = psycopg2.connect(DATABASE_URL)
cur_pp  = conn_pp.cursor()

# ── Phase 0: Schema ─────────────────────────────────────────────────────────
cur_pp.execute(f"""
CREATE TABLE IF NOT EXISTS {TABLE_LABEL_CLASSES} (
    id   SERIAL PRIMARY KEY,
    name TEXT   NOT NULL
);
""")
cur_pp.execute(f"""
CREATE TABLE IF NOT EXISTS {TABLE_GRID_CELLS} (
    id     SERIAL   PRIMARY KEY,
    z_cell BIGINT   NOT NULL,
    level  SMALLINT NOT NULL
);
""")
cur_pp.execute(
    f"CREATE INDEX IF NOT EXISTS idx_gc_z_cell ON {TABLE_GRID_CELLS}(z_cell);"
)
cur_pp.execute(f"""
CREATE TABLE IF NOT EXISTS {TABLE_PRED_PATCHES} (
    id             BIGSERIAL PRIMARY KEY,
    patch_uid      BIGINT    NOT NULL UNIQUE,
    embed_coords   POINT     NOT NULL,
    grid_cell_id   INT       NOT NULL REFERENCES {TABLE_GRID_CELLS}(id),
    event_ts       TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP,
    label_class_id INT       NOT NULL REFERENCES {TABLE_LABEL_CLASSES}(id),
    patch_coords   POINT     NOT NULL
);
""")
cur_pp.execute(
    f"CREATE INDEX IF NOT EXISTS idx_pp_grid_cell_id ON {TABLE_PRED_PATCHES}(grid_cell_id);"
)
conn_pp.commit()

# ── Populate label_classes (once) ───────────────────────────────────────────
cur_pp.execute(f"SELECT COUNT(*) FROM {TABLE_LABEL_CLASSES};")
if cur_pp.fetchone()[0] == 0:
    cur_pp.execute(f"""
        INSERT INTO {TABLE_LABEL_CLASSES} (name)
        SELECT 'class_' || gs FROM generate_series(1, {NUM_LABEL_CLASSES}) gs;
    """)
    conn_pp.commit()
    print(f"  Inserted {NUM_LABEL_CLASSES} label classes.")

# ── Populate grid_cells with all Z-order cells at level Z_LEVEL (once) ──────
# z_cell is the sequential Morton code 0…NUM_GRID_CELLS-1;
# SERIAL id auto-increments so id = z_cell + 1, allowing direct FK lookup.
cur_pp.execute(f"SELECT COUNT(*) FROM {TABLE_GRID_CELLS};")
if cur_pp.fetchone()[0] == 0:
    t_gc = time.time()
    print(f"  Populating {NUM_GRID_CELLS:,} grid cells (Z-order level {Z_LEVEL})...")
    cur_pp.execute(f"""
        INSERT INTO {TABLE_GRID_CELLS} (z_cell, level)
        SELECT gs, {Z_LEVEL} FROM generate_series(0, {NUM_GRID_CELLS - 1}) gs;
    """)
    conn_pp.commit()
    print(f"  Grid cells ready in {time.time() - t_gc:.2f}s.")

# ── Tune for bulk inserts ────────────────────────────────────────────────────
cur_pp.execute("SET maintenance_work_mem = '4GB';")
cur_pp.execute("SET work_mem = '1GB';")
cur_pp.execute("SET synchronous_commit = OFF;")
conn_pp.commit()

# ── Phase 1: Populate pred_patches to 1B rows (skip if already done) ────────
cur_pp.execute(f"SELECT COUNT(*) FROM {TABLE_PRED_PATCHES};")
pp_existing = cur_pp.fetchone()[0]

if pp_existing >= TOTAL_PRED_ROWS:
    print(f"Skipping Phase 1 — {TABLE_PRED_PATCHES} already has {pp_existing:,} rows.")
else:
    rows_to_insert = TOTAL_PRED_ROWS - pp_existing
    print(
        f"\n=== Phase 1: Populate {TABLE_PRED_PATCHES} to {TOTAL_PRED_ROWS:,} rows "
        f"({pp_existing:,} already present, inserting {rows_to_insert:,}) ===\n"
    )

    rows_ins = 0
    batch_n  = 0
    offset   = pp_existing      # patch_uid offset ensures uniqueness across restarts
    t_start  = time.time()

    try:
        cur_pp.execute("BEGIN;")

        while rows_ins < rows_to_insert:
            cur_pp.execute(f"""
                INSERT INTO {TABLE_PRED_PATCHES}
                    (patch_uid, embed_coords, grid_cell_id, event_ts,
                     label_class_id, patch_coords)
                SELECT
                    {offset} + gs                                              AS patch_uid,
                    point(random() * {SPATIAL_EXTENT},
                          random() * {SPATIAL_EXTENT})                         AS embed_coords,
                    (floor(random() * {NUM_GRID_CELLS})::INT + 1)              AS grid_cell_id,
                    CURRENT_TIMESTAMP                                           AS event_ts,
                    (floor(random() * {NUM_LABEL_CLASSES})::INT + 1)           AS label_class_id,
                    point(random() * 10000.0, random() * 10000.0)              AS patch_coords
                FROM generate_series(1, {ANNOT_BATCH_SIZE}) gs;
            """)

            rows_ins += ANNOT_BATCH_SIZE
            offset   += ANNOT_BATCH_SIZE
            batch_n  += 1

            if batch_n % ANNOT_COMMIT_FREQ == 0:
                conn_pp.commit()
                elapsed   = time.time() - t_start
                rate      = rows_ins / elapsed
                remaining = rows_to_insert - rows_ins
                eta_h     = (remaining / rate / 3600) if rate > 0 else 0
                print(
                    f"  {rows_ins:>15,} / {rows_to_insert:,} rows | "
                    f"{elapsed:>8.2f}s | {rate:>10,.0f} rows/s | ETA: {eta_h:.2f}h"
                )
                cur_pp.execute("BEGIN;")

        conn_pp.commit()

    except Exception as exc:
        conn_pp.rollback()
        print(f"Error Phase 1: {exc}")
        raise

    elapsed_p1 = time.time() - t_start
    print(
        f"\nPhase 1 complete: {rows_ins:,} rows inserted in "
        f"{elapsed_p1:.2f}s  ({rows_ins / elapsed_p1:,.0f} rows/sec)"
    )

# Refresh row count for benchmark header
cur_pp.execute(f"SELECT COUNT(*) FROM {TABLE_PRED_PATCHES};")
pp_total = cur_pp.fetchone()[0]

# ── Phase 2: Benchmark — insert 1,000 new pred_patches ──────────────────────
# patch_uid base well above the populated range to avoid UNIQUE conflicts.
# Runs do NOT truncate — the table stays at ~1B rows throughout.
grid_pp  = HierarchicalGridIndexZOrder(cell_size=1.0)
uid_base = pp_total + 1_000_000_000

print(
    f"\n=== Phase 2: Insert {PREDICTION_BATCH_SIZE:,} rows into "
    f"{TABLE_PRED_PATCHES} ({pp_total:,} existing rows, "
    f"B-tree on grid_cell_id, ×{PREDICTION_RUNS} runs) ===\n"
)

pp_latencies = []

for run in range(1, PREDICTION_RUNS + 1):
    # Pre-generate rows — z-cell / coord computation excluded from INSERT timing.
    # grid_cell_id = z_val + 1  (z_val ∈ [0, NUM_GRID_CELLS), id is 1-indexed SERIAL)
    new_rows = []
    for i in range(PREDICTION_BATCH_SIZE):
        x_e   = random.uniform(0, SPATIAL_EXTENT)
        y_e   = random.uniform(0, SPATIAL_EXTENT)
        z_val = grid_pp.point_to_cell(x_e, y_e, Z_LEVEL)
        new_rows.append((
            uid_base + (run - 1) * PREDICTION_BATCH_SIZE + i,          # patch_uid
            f"({x_e},{y_e})",                                           # embed_coords
            z_val + 1,                                                  # grid_cell_id
            f"({random.uniform(0, 10000)},{random.uniform(0, 10000)})", # patch_coords
            random.randint(1, NUM_LABEL_CLASSES),                       # label_class_id
        ))

    t0 = time.time()
    psycopg2.extras.execute_values(
        cur_pp,
        f"""INSERT INTO {TABLE_PRED_PATCHES}
            (patch_uid, embed_coords, grid_cell_id, patch_coords, label_class_id)
            VALUES %s""",
        new_rows,
    )
    conn_pp.commit()
    latency_ms = (time.time() - t0) * 1000
    pp_latencies.append(latency_ms)
    print(
        f"  Run {run:>2}/{PREDICTION_RUNS}: {latency_ms:.2f} ms  "
        f"({PREDICTION_BATCH_SIZE / (latency_ms / 1000):,.0f} rows/sec)"
    )

pp_avg_ms   = sum(pp_latencies) / len(pp_latencies)
pp_min_ms   = min(pp_latencies)
pp_max_ms   = max(pp_latencies)
pp_avg_rate = PREDICTION_BATCH_SIZE / (pp_avg_ms / 1000)

print(f"\n  Table size     : {pp_total:,} rows")
print(f"  Schema         : patch_uid · embed_coords · grid_cell_id (FK) · event_ts · label_class_id (FK) · patch_coords")
print(f"  Index type     : B-tree on grid_cell_id → {TABLE_GRID_CELLS} (Z-order level {Z_LEVEL})")
print(f"  Avg latency    : {pp_avg_ms:.2f} ms")
print(f"  Min latency    : {pp_min_ms:.2f} ms")
print(f"  Max latency    : {pp_max_ms:.2f} ms")
print(f"  Avg rate       : {pp_avg_rate:,.0f} rows/sec")

cur_pp.close()
conn_pp.close()



=== Phase 1: Populate b_annotations_z_index to 1,000,000,000 rows (0 already present, inserting 1,000,000,000) ===

       10,000,000 / 1,000,000,000 rows |    30.38s |    329,172 rows/s | ETA: 0.84h
       20,000,000 / 1,000,000,000 rows |    54.52s |    366,860 rows/s | ETA: 0.74h
       30,000,000 / 1,000,000,000 rows |    84.14s |    356,555 rows/s | ETA: 0.76h
       40,000,000 / 1,000,000,000 rows |   114.13s |    350,489 rows/s | ETA: 0.76h
       50,000,000 / 1,000,000,000 rows |   144.03s |    347,151 rows/s | ETA: 0.76h
       60,000,000 / 1,000,000,000 rows |   173.57s |    345,685 rows/s | ETA: 0.76h
       70,000,000 / 1,000,000,000 rows |   204.50s |    342,306 rows/s | ETA: 0.75h
       80,000,000 / 1,000,000,000 rows |   236.81s |    337,828 rows/s | ETA: 0.76h
       90,000,000 / 1,000,000,000 rows |   268.66s |    335,000 rows/s | ETA: 0.75h
      100,000,000 / 1,000,000,000 rows |   300.48s |    332,798 rows/s | ETA: 0.75h
      110,000,000 / 1,000,000,000 rows |   

KeyboardInterrupt: 